# NYC Airbnb 2026 Price Prediction

Objective: build a regression model for predicting Airbnb listing `price` in New York City using the 2026 market dataset at `data/airbnb_2026_market.csv` which combines the listings of January and February.

# Background
I had done a project for AMS 317 (Linear Regression Analysis) where my group and I looked into 2019 AirBnb dataset for New York City. We had done a complete data analysis, EDA, used ANOVA to measure the question "Does different types of rooms vs the number of reviews".

I wanted to do a similar analysis on Airbnb dataset but on a better idea. Instead of just conducting an analysis and ANOVA. I wanted to build a model and maybe answer some business question. I am also a frequent Airbnb user when I go on trips and lots of times, I found myself asking, "Is this price reasonable in comparision to the surrounding Airbnb based on similar metrics and characteristics?" So I decided to look into the 2026 NYC Airbnb Dataset, and decided to explore the following question with real world messy data from https://insideairbnb.com/.

# Business question
How can we predict a fair nightly price for a New York City Airbnb listing using listing and review characteristics, and identify whether the pricing is above or below market?



Before even examining the data, I like to understand the problem first. 
* `What are we predicting?` : The Price
* `What are the type of data are we training on?`: offline batch learning because the dataset is static up until April of 2026.
* `What type of model should we use?`: Regression because we are predicting the price.
* `What metrics are we measuring performance by?`: 
* `Does the performance align with the business question?`: 

## Workflow

1. Load the raw 2026 market dataset.
2. Inspect shape, schema, duplicates, missingness, and target coverage.
3. Identify obvious data-quality issues that affect price modeling.
4. Define the next cleaning steps before moving into EDA and modeling.

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'airbnb_2026_market.csv'

In [23]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
df.head()

Shape: 38,076 rows x 19 columns


,id,price,room_type,neighbourhood,neighbourhood_group,latitude,longitude,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,reviews_last_30d,reviews_last_90d,reviews_last_365d,days_since_last_review,is_licensed,has_reviews
0,2539,NaN,Private room,Kensington,Brooklyn,40.645,-73.972,30.000,9,0.070,7,364,0,0,0,0,"2,661.000",0,1
1,2595,240.000,Entire home/apt,Midtown,Manhattan,40.754,-73.986,30.000,47,0.240,3,319,0,0,0,0,"1,320.000",0,1
2,6848,139.000,Entire home/apt,Williamsburg,Brooklyn,40.709,-73.953,30.000,197,0.975,1,270,4,0,0,4,75.000,0,1
3,6872,59.000,Private room,East Harlem,Manhattan,40.801,-73.943,30.000,2,0.050,2,83,1,0,0,1,116.000,0,1
4,6990,73.000,Private room,East Harlem,Manhattan,40.788,-73.948,30.000,251,1.275,1,302,4,0,2,4,20.000,0,1


## Initial Data Inspection

In [33]:
schema_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
    'missing_count': df.isna().sum().values,
    'missing_pct': (df.isna().mean() * 100).round(2).values,
    'unique_values': df.nunique(dropna=True).values,
}).sort_values(['missing_pct', 'column'], ascending=[False, True]).reset_index(drop=True)

schema_summary

,column,dtype,non_null_count,missing_count,missing_pct,unique_values
0,price,float64,21415,16661,43.760,1106
1,days_since_last_review,float64,25211,12865,33.790,3485
2,reviews_per_month,float64,25981,12095,31.770,1543
3,availability_365,int64,38076,0,0.000,366
4,calculated_host_listings_count,int64,38076,0,0.000,87
5,has_reviews,int64,38076,0,0.000,2
6,id,int64,38076,0,0.000,38076
7,is_licensed,int64,38076,0,0.000,2
8,latitude,float64,38076,0,0.000,23288
9,longitude,float64,38076,0,0.000,21063


In [34]:
duplicate_and_coverage = pd.Series({
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_ids': int(df['id'].duplicated().sum())
})

duplicate_and_coverage

duplicate_rows    0
duplicate_ids     0
dtype: int64

In [35]:
numeric_summary = df.describe().T
numeric_summary

,count,mean,std,min,25%,50%,75%,max
id,"38,076.000","513,220,995,058,672,064.000","575,944,947,165,107,968.000","2,539.000","22,327,326.750","52,859,573.000","1,041,552,379,817,649,152.000","1,598,134,939,144,730,880.000"
price,"21,415.000",519.623,"3,658.431",9.000,90.000,154.000,269.000,"50,138.000"
latitude,"38,076.000",40.729,0.056,40.505,40.689,40.727,40.762,40.912
longitude,"38,076.000",-73.948,0.055,-74.252,-73.984,-73.955,-73.928,-73.712
minimum_nights,"38,075.000",28.590,29.116,1.000,30.000,30.000,30.000,"1,124.000"
number_of_reviews,"38,076.000",27.473,71.797,0.000,0.000,3.000,22.000,"4,095.000"
reviews_per_month,"25,981.000",0.829,1.903,0.010,0.075,0.255,0.890,120.865
calculated_host_listings_count,"38,076.000",68.263,219.297,1.000,1.000,2.000,10.000,"1,197.000"
availability_365,"38,076.000",168.895,148.203,0.000,0.000,165.000,326.000,365.000
number_of_reviews_ltm,"38,076.000",4.239,19.724,0.000,0.000,0.000,1.000,"1,639.000"


In [36]:
categorical_summary = pd.DataFrame({
    'column': df.select_dtypes(include='object').columns,
    'unique_values': [df[col].nunique(dropna=True) for col in df.select_dtypes(include='object').columns],
    'top_value': [df[col].mode(dropna=True).iat[0] if not df[col].mode(dropna=True).empty else np.nan for col in df.select_dtypes(include='object').columns],
    'top_value_freq': [df[col].value_counts(dropna=True).iloc[0] if not df[col].value_counts(dropna=True).empty else np.nan for col in df.select_dtypes(include='object').columns],
})

categorical_summary

/var/folders/7d/s386vfvx5wd6mh0bc0nhmgqr0000gn/T/ipykernel_12929/1273753840.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  'column': df.select_dtypes(include='object').columns,
/var/folders/7d/s386vfvx5wd6mh0bc0nhmgqr0000gn/T/ipykernel_12929/1273753840.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://panda

,column,unique_values,top_value,top_value_freq
0,room_type,4,Entire home/apt,20369
1,neighbourhood,224,Bedford-Stuyvesant,2691
2,neighbourhood_group,5,Manhattan,17219


In [37]:
room_type_counts = df['room_type'].value_counts(dropna=False).rename_axis('room_type').reset_index(name='listing_count')
borough_counts = df['neighbourhood_group'].value_counts(dropna=False).rename_axis('neighbourhood_group').reset_index(name='listing_count')

top_neighbourhoods = (
    df['neighbourhood']
    .value_counts(dropna=False)
    .head(15)
    .rename_axis('neighbourhood')
    .reset_index(name='listing_count')
)

room_type_counts, borough_counts, top_neighbourhoods

(         room_type  listing_count
 0  Entire home/apt          20369
 1     Private room          17026
 2       Hotel room            367
 3      Shared room            314,
   neighbourhood_group  listing_count
 0           Manhattan          17219
 1            Brooklyn          13745
 2              Queens           5556
 3               Bronx           1175
 4       Staten Island            381,
          neighbourhood  listing_count
 0   Bedford-Stuyvesant           2691
 1              Midtown           2159
 2         Williamsburg           2142
 3               Harlem           1763
 4       Hell's Kitchen           1631
 5      Upper West Side           1578
 6      Upper East Side           1511
 7             Bushwick           1500
 8        Crown Heights           1151
 9         East Village           1022
 10             Chelsea            910
 11         East Harlem            715
 12  Financial District            682
 13     Lower East Side            664
 14       

In [39]:
missing_summary = (
    pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
    })
    .sort_values(['missing_count', 'missing_pct'], ascending=False)
)

missing_summary[missing_summary['missing_count'] > 0]

,missing_count,missing_pct
price,16661,43.760
days_since_last_review,12865,33.790
reviews_per_month,12095,31.770
minimum_nights,1,0.000


In [40]:
price_summary = pd.Series({
    'non_missing_price_rows': int(df['price'].notna().sum()),
    'missing_price_rows': int(df['price'].isna().sum()),
    'missing_price_pct': round(df['price'].isna().mean() * 100, 2),
    'min_price': df['price'].min(),
    'median_price': df['price'].median(),
    'mean_price': df['price'].mean(),
    'p90_price': df['price'].quantile(0.90),
    'p95_price': df['price'].quantile(0.95),
    'p99_price': df['price'].quantile(0.99),
    'max_price': df['price'].max(),
})

price_summary.to_frame('value')

,value
non_missing_price_rows,"21,415.000"
missing_price_rows,"16,661.000"
missing_price_pct,43.760
min_price,9.000
median_price,154.000
mean_price,519.623
p90_price,428.000
p95_price,611.300
p99_price,"2,266.160"
max_price,"50,138.000"


In [42]:
review_consistency_checks = pd.Series({
    'has_reviews_flag_mismatches': int((df['has_reviews'] != (df['number_of_reviews'] > 0).astype(int)).sum()),
    'no_reviews_but_recent_review_days_present': int(((df['has_reviews'] == 0) & df['days_since_last_review'].notna()).sum()),
    'positive_reviews_but_missing_days_since_last_review': int(((df['number_of_reviews'] > 0) & df['days_since_last_review'].isna()).sum()),
    'non_positive_price_rows': int(df['price'].fillna(1).le(0).sum()),
    'missing_minimum_nights_rows': int(df['minimum_nights'].isna().sum()),
})

review_consistency_checks.to_frame('count')

,count
has_reviews_flag_mismatches,0
no_reviews_but_recent_review_days_present,123
positive_reviews_but_missing_days_since_last_review,893
non_positive_price_rows,0
missing_minimum_nights_rows,1
